# Magnitude-Class Variance Decomposition (η²) and R²_fit Comparison

This notebook reproduces, from scratch and step by step, the η² table used in the MLR Section 5.1
discussion text (the variance ratio explainable by "magnitude class" for 12 physical properties,
and its comparison with the MLR model's R²_fit value).

**What we are doing, and why:**
1. We group each molecule by its ACTUAL molecular formula (different formula = different isomer
   class = different magnitude class). This formula is NOT DERIVED BY ANY RULE/CALCULATION: the
   actual molecular formula of each of the 67 molecules in the dataset (determined by inspecting
   its structure) has been written ONE BY ONE, BY HAND, into the table in Section 2. Grouping is
   done directly from the values in this fixed table — MW or any other numerical/algorithmic
   derivation is not consulted.
2. For each physical property, we calculate, via one-way ANOVA variance decomposition, how much of
   the total variance (η²) can be explained SOLELY by this magnitude-class information (without
   any regard to the isomer's own branching).
3. We know that in the 12 winning equations in the MLR results report (Eq. 11–22), the last two remaining
   descriptors are always the same (Per(A), A(2,2)); for the eight properties where the magnitude
   classes are mostly NOT single-molecule, we independently recompute R²_fit using an OLS fit with
   these same two descriptors, and compare it with η² (Difference = R²_fit − η²).

**Important limitation:** This notebook does NOT re-run the VIF elimination / Topliss–Costello
step — those steps are already done in `MLR_Analysis.ipynb`. Here, only the R² of the
FINAL model known from the tables in the MLR results report (Per(A), A(2,2)) is independently rebuilt as
a validation check.

## 1. Libraries and data file

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

DATA_FILE = "Alcohol_Data.xlsx"  # must be in the same folder as this notebook
MOLECULE_NAME_COLUMN = "Aliphatic Alcohol"

## 2. Molecular formula table — ENTIRELY HAND-WRITTEN

No rule/calculation/parsing is involved: the ACTUAL molecular formula of each of the 67 molecules
in the dataset (determined by inspecting its structure) has been written here by hand, one by one.
Grouping is done directly from the values in this fixed table — MW or any other
numerical/algorithmic derivation is not consulted.

In [ ]:
MOLECULAR_FORMULA_TABLE = {
    "Methanol": "CH4O",
    "Ethanol": "C2H6O",
    "1-Propanol": "C3H8O",
    "2-Propanol": "C3H8O",
    "1-Butanol": "C4H10O",
    "2-Butanol": "C4H10O",
    "2-M-1-Propanol": "C4H10O",
    "2-M-2-Propanol": "C4H10O",
    "1-Pentanol": "C5H12O",
    "2-Pentanol": "C5H12O",
    "3-Pentanol": "C5H12O",
    "2-M-1-Butanol": "C5H12O",
    "3-M-1-Butanol": "C5H12O",
    "2-M-2-Butanol": "C5H12O",
    "3-M-2-Butanol": "C5H12O",
    "2,2-MM-1-Propanol": "C5H12O",
    "1-Hexanol": "C6H14O",
    "2-Hexanol": "C6H14O",
    "3-Hexanol": "C6H14O",
    "2-M-1-Pentanol": "C6H14O",
    "3-M-1-Pentanol": "C6H14O",
    "4-M-1-Pentanol": "C6H14O",
    "2-M-2-Pentanol": "C6H14O",
    "3-M-2-Pentanol": "C6H14O",
    "4-M-2-Pentanol": "C6H14O",
    "2-M-3-Pentanol": "C6H14O",
    "3-M-3-Pentanol": "C6H14O",
    "2-E-1-Butanol": "C6H14O",
    "2,2-M-M-1-Butanol": "C6H14O",
    "3,3-M-M-1-Butanol": "C6H14O",
    "2,3-M-M-2-Butanol": "C6H14O",
    "3,3-M-M-2-Butanol": "C6H14O",
    "1-heptanol": "C7H16O",
    "3-heptanol": "C7H16O",
    "4-heptanol": "C7H16O",
    "2-M-2-hexanol": "C7H16O",
    "3-M-3-hexanol": "C7H16O",
    "3-E-3-pentanol": "C7H16O",
    "2,2-M-M-3-pentanol": "C7H16O",
    "2,3-M-M-3-pentanol": "C7H16O",
    "2,4-M-M-3-pentanol": "C7H16O",
    "1-octanol": "C8H18O",
    "2-octanol": "C8H18O",
    "2-E-1-hexanol": "C8H18O",
    "1-nonanol": "C9H20O",
    "2-nonanol": "C9H20O",
    "3-nonanol": "C9H20O",
    "4-nonanol": "C9H20O",
    "5-nonanol": "C9H20O",
    "2,6-M-M-4-heptanol": "C9H20O",
    "3,5-M-M-4-heptanol": "C9H20O",
    "3,5,5-M-M-M-1-hexanol": "C9H20O",
    "1-decanol": "C10H22O",
    "5-M-1-Hexanol": "C7H16O",
    "5-M-2-Hexanol": "C7H16O",
    "2-Heptanol": "C7H16O",
    "2-M-3-Heptanol": "C8H18O",
    "3-M-4-Heptanol": "C8H18O",
    "3-Octanol": "C8H18O",
    "4-Octanol": "C8H18O",
    "3-M-2-Heptanol": "C8H18O",
    "5-M-2-Heptanol": "C8H18O",
    "6-M-1-Heptanol": "C8H18O",
    "4-M-1-Heptanol": "C8H18O",
    "2,2,4-MMM-3-Pentanol": "C8H18O",
    "2-M-2-Octanol": "C9H20O",
}

## 3. Property → winning group mapping

Here we define, for each physical property, the winning descriptor group (and therefore the
corresponding Excel sheet) indicated in the MLR results report with the "For [Group] with/without
hydrogen," label.

In [ ]:
PROPERTY_GROUP_MAPPING = {
    "bp - Boiling Point (°C)":          "CW-mix_with_H",
    "MW - Molecular Weight":              "CW-mix_with_H",
    "nD - Refractive Index":                "CW-I_without_H",
    "epsilon - Dielectric Constant":        "CW-mix_with_H",
    "Tc - Critical Temperature (K)":           "CW-mix_with_H",
    "Vc - Critical Volume (cm3/mol)":        "CW-mix_with_H",
    "Pc - Critical Pressure (MPa)":           "CW-mix_with_H",
    "logP":                               "CW-mix_with_H",
    "vp - Vapor Pressure (kPa)":           "CW-I_without_H",
    "eta - Viscosity (mPa.s)":            "CW-mix_with_H",
    "gamma - Surface Tension (mN/m)":      "CW-II_with_H",
    "lambda - Thermal Conductivity (W/m.K)": "CW-II_without_H",
}

# The difference (R^2 - eta^2) is calculated only for these eight properties,
# for which the magnitude classes are mostly multi-member. vp, eta, gamma, lambda
# are EXCLUDED because of their small n's and the weight of single-molecule
# groups (you can verify this yourself below in Section 6 by looking at the
# group sizes).
PROPERTIES_FOR_DIFFERENCE_CALCULATION = {
    "bp - Boiling Point (°C)",
    "MW - Molecular Weight",
    "nD - Refractive Index",
    "epsilon - Dielectric Constant",
    "Tc - Critical Temperature (K)",
    "Vc - Critical Volume (cm3/mol)",
    "Pc - Critical Pressure (MPa)",
    "logP",
}

## 4. η² function

Groups molecules (using the hand-written table from Section 2) by their actual molecular
formulas, and calculates η² = SS_between_groups / SS_total using one-way ANOVA logic. MW does not
enter this calculation at all, and the formula is not computed by any rule either — it is read
directly from the fixed table.

In [ ]:
def calculate_eta_squared(molecule_name_series, property_series):
    """Groups molecules by their ACTUAL molecular formulas (hand-written) from
    MOLECULAR_FORMULA_TABLE and calculates eta^2 (the between-magnitude-class
    variance ratio) using one-way ANOVA. MW does NOT enter this calculation
    at all, and the formula is not computed by any rule either - it is read
    directly from the fixed table."""
    y_numeric = pd.to_numeric(property_series, errors="coerce")  # turns empty cells like '-' into NaN
    df = pd.DataFrame({"molecule_name": molecule_name_series, "y": y_numeric}).dropna()

    df["molecular_formula"] = df["molecule_name"].map(MOLECULAR_FORMULA_TABLE)
    missing = df.loc[df["molecular_formula"].isna(), "molecule_name"].unique().tolist()
    if missing:
        raise ValueError(
            f"There are molecules with no formula defined in MOLECULAR_FORMULA_TABLE: "
            f"{missing} - first add the formula for these molecules to the table by hand."
        )

    overall_mean = df["y"].mean()
    ss_total = ((df["y"] - overall_mean) ** 2).sum()

    ss_between_groups = 0.0
    for _, group in df.groupby("molecular_formula"):
        ss_between_groups += len(group) * (group["y"].mean() - overall_mean) ** 2

    eta2 = ss_between_groups / ss_total
    group_sizes = sorted(df.groupby("molecular_formula").size().tolist(), reverse=True)
    return eta2, len(df), group_sizes

## 5. R²_fit function

Reads the Per(A) and A(2,2) descriptors from the corresponding CW group sheet and calculates
R²_fit with OLS (the same descriptor set as the final model of the winning equation in the MLR results report).

In [ ]:
def calculate_r2_fit(all_properties_df, group_sheet_name, property_column):
    """... If a molecule name is NOT FOUND in the CW group sheet (spelling
    difference, whitespace, etc.), raises a ValueError instead of SILENTLY
    skipping it - to prevent R^2_fit from being calculated on fewer molecules
    than expected without anyone noticing."""
    group_df = pd.read_excel(DATA_FILE, sheet_name=group_sheet_name)

    all_copy = all_properties_df[[MOLECULE_NAME_COLUMN, property_column]].copy()
    all_copy[property_column] = pd.to_numeric(all_copy[property_column], errors="coerce")
    all_copy = all_copy.dropna(subset=[property_column])

    merged = all_copy.merge(
        group_df[["Solvent", "Per(A)", "A(2,2)"]],
        left_on=MOLECULE_NAME_COLUMN,
        right_on="Solvent",
        how="inner",
    )

    if len(merged) != len(all_copy):
        missing = sorted(set(all_copy[MOLECULE_NAME_COLUMN]) - set(merged[MOLECULE_NAME_COLUMN]))
        raise ValueError(
            f"For {property_column!r}, there are {len(missing)} molecules not matched in sheet "
            f"{group_sheet_name!r}: {missing} - R^2_fit was not silently calculated with "
            "missing molecules, first fix this name mismatch."
        )

    X = sm.add_constant(merged[["Per(A)", "A(2,2)"]].astype(float))
    y = merged[property_column].astype(float)
    model = sm.OLS(y, X).fit()
    return model.rsquared, len(merged)

## 6. Read the data and calculate for all properties

In [ ]:
all_df = pd.read_excel(DATA_FILE, sheet_name="All_Properties")
molecule_name_series = all_df[MOLECULE_NAME_COLUMN]

rows = []
for property_column, group_sheet in PROPERTY_GROUP_MAPPING.items():
    eta2, n, group_sizes = calculate_eta_squared(molecule_name_series, all_df[property_column])
    singleton_count = sum(1 for s in group_sizes if s == 1)

    if property_column in PROPERTIES_FOR_DIFFERENCE_CALCULATION:
        r2_fit, n_model = calculate_r2_fit(all_df, group_sheet, property_column)
        difference = r2_fit - eta2
    else:
        r2_fit, difference = None, None

    rows.append({
        "property": property_column,
        "n": n,
        "eta2": round(eta2, 4),
        "within_isomer_ratio": round(1 - eta2, 4),
        "group_count": len(group_sizes),
        "singleton_group_count": singleton_count,
        "group_sizes": group_sizes,
        "R2_fit": round(r2_fit, 4) if r2_fit is not None else None,
        "diff_R2_minus_eta2": round(difference, 4) if difference is not None else None,
    })

results_df = pd.DataFrame(rows).sort_values("eta2", ascending=False)
results_df

## 7. Let's also look at the group sizes (how many singleton groups per property)

In [ ]:
for _, row in results_df.iterrows():
    print(f"{row['property']:42s} group sizes: {row['group_sizes']}")

## 8. Save the results as CSV

In [ ]:
results_df.to_csv("eta_squared_analysis_results.csv", index=False, encoding="utf-8-sig")
print("Saved: eta_squared_analysis_results.csv")